# EXP 01 — Shared-Feature Match-Level Baseline dengan CatBoost, Symmetry-Aware Feature Engineering, dan Metric-Aware Joint Score Decoding

---

## Tujuan Eksperimen

Ini adalah **baseline modeling pertama yang benar-benar feasible terhadap test set**.

Berdasarkan temuan EXP 00:
- Train memiliki 47 kolom, test hanya 20 — ada **27 fitur train-only** yang tidak tersedia saat inference.
- Satu pertandingan = dua row; representasi match-level canonical sudah valid.
- Metrik kompetisi (**AW-MAE**) sangat sensitif terhadap kesalahan outcome.
- Terdapat **50 unseen teams** di test.
- Ada distribution shift pada gender dan tournament.

### Apa yang dilakukan EXP 01?

| Komponen | Deskripsi |
|---|---|
| **Shared Features Only** | Hanya fitur yang ada di train & test |
| **Match-Level Modeling** | Satu baris per pertandingan |
| **Baseline A** | Direct goals regression + rounding |
| **Baseline B** | Outcome + GD + Total decomposition + joint decoder |
| **Model** | CatBoost (native categorical, robust missing) |
| **Evaluasi** | Offline AW-MAE pada temporal validation |

### Hipotesis
1. Model shared-features mengalahkan baseline naïf 1-1
2. Decomposition + decoder lebih cocok dengan AW-MAE dibanding direct rounding
3. CatBoost tepat untuk data tabular mixed-type dengan unseen categories
4. Joint integer decoding lebih baik dari rounding biasa

## 01. Setup, Seed, dan Konfigurasi Path

In [1]:
import os
import json
import math
import random
import warnings
from pathlib import Path
from itertools import product
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostRegressor, CatBoostClassifier, Pool

print("[INFO] Semua library berhasil di-import.")

[INFO] Semua library berhasil di-import.


In [2]:
# === KONFIGURASI GLOBAL ===
SEED = 42

TRAIN_PATH = "../data/train.csv"
TEST_PATH  = "../data/test.csv"
SAMPLE_SUB_PATH = "../data/sample submission.csv"
META_PATH  = "../data/metadata.txt"

OUT_ROOT = "../outputs/exp01_shared_feature_catboost_metric_aware"
FIG_DIR  = f"{OUT_ROOT}/figures"
PRED_DIR = f"{OUT_ROOT}/predictions"
SUB_DIR  = f"{OUT_ROOT}/submissions"
SUM_DIR  = f"{OUT_ROOT}/summaries"

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
warnings.filterwarnings("ignore")

print(f"[INFO] SEED = {SEED}")

[INFO] SEED = 42


In [3]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[INFO] Seed set to {seed}")

seed_everything(SEED)

[INFO] Seed set to 42


In [4]:
# Buat folder output
for d in [FIG_DIR, PRED_DIR, SUB_DIR, SUM_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"[OK] {d}")

[OK] ../outputs/exp01_shared_feature_catboost_metric_aware/figures
[OK] ../outputs/exp01_shared_feature_catboost_metric_aware/predictions
[OK] ../outputs/exp01_shared_feature_catboost_metric_aware/submissions
[OK] ../outputs/exp01_shared_feature_catboost_metric_aware/summaries


### Catatan Setup
- Notebook ini **standalone** — tidak bergantung pada state notebook lain.
- Helper penting dari EXP 00 ditulis ulang di sini.
- Folder output dibuat eksplisit untuk menyimpan artifact eksperimen.

## 02. Validasi File Input

In [5]:
FILE_PATHS = {
    "train": TRAIN_PATH,
    "test": TEST_PATH,
    "sample_submission": SAMPLE_SUB_PATH,
    "metadata": META_PATH,
}

def validate_input_files(file_paths: dict) -> None:
    missing = []
    for name, path in file_paths.items():
        if not Path(path).exists():
            missing.append((name, path))
    if missing:
        msg = "File TIDAK ditemukan:\n" + "\n".join(
            f"  - {n}: {p}" for n, p in missing
        )
        raise FileNotFoundError(msg)
    print("[OK] Semua file input ditemukan.")

validate_input_files(FILE_PATHS)

[OK] Semua file input ditemukan.


## 03. Load Data dan Helper Inti dari EXP 00

In [6]:
# --- Load data ---
train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
test  = pd.read_csv(TEST_PATH,  parse_dates=["date"])
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Sample: {sample_sub.shape}")

Train : (78772, 47)
Test  : (42422, 20)
Sample: (42422, 3)


In [7]:
# ============================================================
# HELPER FUNCTIONS — ditulis ulang dari EXP 00 agar standalone
# ============================================================

def build_match_level(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    """Konversi row-level → match-level. Canonical: team_a = alfabet pertama."""
    counts = df.groupby("match_id").size()
    bad = counts[counts != 2]
    if len(bad) > 0:
        raise ValueError(f"{len(bad)} match_id tanpa tepat 2 rows")

    df_sorted = df.sort_values(["match_id", "team"]).reset_index(drop=True)
    row_a = df_sorted.iloc[0::2].reset_index(drop=True)
    row_b = df_sorted.iloc[1::2].reset_index(drop=True)
    assert (row_a["match_id"].values == row_b["match_id"].values).all()

    result = pd.DataFrame()
    for col in ["match_id", "date", "gender", "tournament", "venue_country", "neutral"]:
        if col in df.columns:
            result[col] = row_a[col].values
    for col in ["altitude_venue", "temperature_venue"]:
        if col in df.columns:
            result[col] = row_a[col].values

    result["team_a"] = row_a["team"].values
    result["team_b"] = row_b["team"].values
    if "is_home" in df.columns:
        result["team_a_is_home"] = row_a["is_home"].values
        result["team_b_is_home"] = row_b["is_home"].values

    sym = {"confederation_team":"confederation","population_team":"population",
           "gdp_per_capita_team":"gdp_per_capita","distance_travel_team":"distance_travel"}
    for orig, suf in sym.items():
        if orig in df.columns:
            result[f"team_a_{suf}"] = row_a[orig].values
            result[f"team_b_{suf}"] = row_b[orig].values

    if is_train and "team_goals" in df.columns:
        result["team_a_goals"] = row_a["team_goals"].values
        result["team_b_goals"] = row_a["opp_goals"].values

    # Train-only perf features (tetap dimasukkan jika ada)
    perf = {"elo_team":"elo","rank_team":"rank","rank_missing_team":"rank_missing",
            "team_points_last5":"points_last5","team_gd_last5":"gd_last5",
            "team_points_last10":"points_last10","team_avg_goals_last5":"avg_goals_last5",
            "team_avg_conceded_last5":"avg_conceded_last5","team_win_rate_last10":"win_rate_last10",
            "days_since_last_match_team":"days_since_last_match"}
    for orig, suf in perf.items():
        if orig in df.columns:
            result[f"team_a_{suf}"] = row_a[orig].values
            result[f"team_b_{suf}"] = row_b[orig].values
    opp_perf = {"elo_opponent":"opp_elo","rank_opponent":"opp_rank",
                "rank_missing_opp":"opp_rank_missing","opp_points_last5":"opp_points_last5",
                "opp_gd_last5":"opp_gd_last5","opp_points_last10":"opp_points_last10",
                "opp_avg_goals_last5":"opp_avg_goals_last5",
                "opp_avg_conceded_last5":"opp_avg_conceded_last5",
                "opp_win_rate_last10":"opp_win_rate_last10",
                "days_since_last_match_opp":"opp_days_since_last_match"}
    for orig, suf in opp_perf.items():
        if orig in df.columns:
            result[f"team_a_{suf}"] = row_a[orig].values
            result[f"team_b_{suf}"] = row_b[orig].values
    for col in ["points_last5_diff","gd_last5_diff","rank_diff","h2h_points_last5","h2h_gd_last5"]:
        if col in df.columns:
            result[f"team_a_{col}"] = row_a[col].values
            result[f"team_b_{col}"] = row_b[col].values
    return result


def match_predictions_to_submission(
    test_row_df, pred_match_df,
    canonical_team_a_col="team_a", canonical_team_b_col="team_b",
    pred_a_col="pred_team_a_goals", pred_b_col="pred_team_b_goals",
):
    """Konversi prediksi match-level → submission row-level."""
    merge_cols = ["match_id", canonical_team_a_col, canonical_team_b_col,
                  pred_a_col, pred_b_col]
    merged = test_row_df[["Id","match_id","team"]].merge(
        pred_match_df[merge_cols], on="match_id", how="left",
    )
    is_a = merged["team"] == merged[canonical_team_a_col]
    merged["team_goals"] = np.where(is_a, merged[pred_a_col], merged[pred_b_col])
    merged["opp_goals"]  = np.where(is_a, merged[pred_b_col], merged[pred_a_col])
    return merged[["Id","team_goals","opp_goals"]].copy()


# --- AW-MAE Evaluator ---
EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50

def _outcome(a, b):
    a, b = float(a), float(b)
    return 1 if a > b else (0 if a == b else -1)

def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()
    if "fifa world cup" in t or t == "world cup":
        return 2.00
    elif "afc championship" in t or "afc asian cup" in t or "asian cup" in t:
        return 1.80
    elif "friendly" in t:
        return 0.96
    else:
        return 1.20

def official_match_loss(y_team_true, y_opp_true, y_team_pred, y_opp_pred):
    yt, yo = float(y_team_true), float(y_opp_true)
    pt, po = float(y_team_pred), float(y_opp_pred)
    mae = (abs(yt - pt) + abs(yo - po)) / 2.0
    exact = 1 if (yt == pt and yo == po) else 0
    oc = 1 if _outcome(yt, yo) == _outcome(pt, po) else 0
    gdc = 1 if (yt - yo) == (pt - po) else 0
    penalty = EXACT_PENALTY*(1-exact) + OUTCOME_PENALTY*(1-oc) + GD_PENALTY*(1-gdc)
    mult = 1.0 if oc else WRONG_OUTCOME_MULTIPLIER
    return ((mae + penalty) * mult) ** NONLINEAR_POWER

def awmae_score(y_team_true, y_opp_true, y_team_pred, y_opp_pred, tournaments):
    """Loop-based AW-MAE (reference implementation)."""
    yt = np.asarray(y_team_true, dtype=float)
    yo = np.asarray(y_opp_true, dtype=float)
    pt = np.asarray(y_team_pred, dtype=float)
    po = np.asarray(y_opp_pred, dtype=float)
    n = len(yt)
    tlist = tournaments.tolist() if hasattr(tournaments, "tolist") else list(tournaments)
    wl, wt = 0.0, 0.0
    for i in range(n):
        loss = official_match_loss(yt[i], yo[i], pt[i], po[i])
        w = get_tournament_weight(str(tlist[i]))
        wl += loss * w
        wt += w
    return wl / wt if wt > 0 else 0.0

def awmae_score_fast(y_a_true, y_b_true, y_a_pred, y_b_pred, weights):
    """Vectorized AW-MAE for fast grid search."""
    yt_a = np.asarray(y_a_true, dtype=float)
    yt_b = np.asarray(y_b_true, dtype=float)
    yp_a = np.asarray(y_a_pred, dtype=float)
    yp_b = np.asarray(y_b_pred, dtype=float)
    w = np.asarray(weights, dtype=float)
    mae = (np.abs(yt_a - yp_a) + np.abs(yt_b - yp_b)) / 2.0
    exact = ((yt_a == yp_a) & (yt_b == yp_b)).astype(float)
    sign_t = np.sign(yt_a - yt_b)
    sign_p = np.sign(yp_a - yp_b)
    oc = (sign_t == sign_p).astype(float)
    gdc = ((yt_a - yt_b) == (yp_a - yp_b)).astype(float)
    penalty = 0.30*(1-exact) + 0.25*(1-oc) + 0.15*(1-gdc)
    mult = np.where(oc, 1.0, 1.5)
    loss = ((mae + penalty) * mult) ** 1.5
    return np.sum(loss * w) / np.sum(w) if np.sum(w) > 0 else 0.0

def make_time_based_holdout(train_match, valid_fraction=0.2):
    """Split temporal: 20% match terakhir jadi validation."""
    df = train_match.sort_values("date").reset_index(drop=True)
    n = len(df)
    idx = int(n * (1 - valid_fraction))
    tr = df.iloc[:idx].reset_index(drop=True)
    vl = df.iloc[idx:].reset_index(drop=True)
    print(f"Train fold: {len(tr):,} matches  ({tr['date'].min()} -> {tr['date'].max()})")
    print(f"Valid fold: {len(vl):,} matches  ({vl['date'].min()} -> {vl['date'].max()})")
    return tr, vl

print("[OK] Semua helper berhasil didefinisikan.")

[OK] Semua helper berhasil didefinisikan.


### Catatan Helper
Seluruh helper inti dari EXP 00 ditulis ulang di notebook ini agar **standalone**.
Ditambahkan `awmae_score_fast` (versi vectorized) untuk efisiensi grid search decoder.

## 04. Cleaning Awal dan Shared-Feature Enforcement

Cleaning pada EXP 01 masih **konservatif**:
- Sentinel `altitude_venue == -9999` → `NaN`
- Pastikan kolom kategorikal bertipe string
- Belum ada imputasi agresif — CatBoost menangani missing secara native

In [8]:
# Definisi eksplisit kolom
EXPECTED_SHARED_COLS = [
    "Id", "match_id", "date", "gender", "team", "opponent",
    "is_home", "neutral", "tournament", "venue_country",
    "confederation_team", "confederation_opp",
    "population_team", "population_opp",
    "gdp_per_capita_team", "gdp_per_capita_opp",
    "altitude_venue", "distance_travel_team", "distance_travel_opp",
    "temperature_venue",
]
TARGET_COLS = ["team_goals", "opp_goals"]
ID_COLS = ["Id", "match_id"]

# Assert shared cols tersedia
for col in EXPECTED_SHARED_COLS:
    assert col in train.columns, f"{col} tidak di train!"
    assert col in test.columns,  f"{col} tidak di test!"
print("[OK] Semua shared columns tersedia di train dan test.")

[OK] Semua shared columns tersedia di train dan test.


In [9]:
# Bersihkan sentinel altitude_venue == -9999
SENTINEL = -9999
for name, df in [("train", train), ("test", test)]:
    if "altitude_venue" in df.columns:
        n_sent = (df["altitude_venue"] == SENTINEL).sum()
        df.loc[df["altitude_venue"] == SENTINEL, "altitude_venue"] = np.nan
        print(f"  {name}: {n_sent:,} sentinel altitude_venue -> NaN")

  train: 766 sentinel altitude_venue -> NaN
  test: 296 sentinel altitude_venue -> NaN


In [10]:
# Pastikan kolom kategorikal bertipe string
CAT_RAW_COLS = ["team","opponent","tournament","venue_country","gender",
                "confederation_team","confederation_opp"]
for col in CAT_RAW_COLS:
    for df in [train, test]:
        if col in df.columns:
            df[col] = df[col].astype(str)
print("[OK] Kolom kategorikal aman sebagai string.")

[OK] Kolom kategorikal aman sebagai string.


## 05. Bangun Match-Level Canonical Data

In [11]:
train_match = build_match_level(train, is_train=True)
test_match  = build_match_level(test,  is_train=False)

print(f"train_match: {train_match.shape}")
print(f"test_match : {test_match.shape}")
assert len(train_match) == train["match_id"].nunique()
assert len(test_match)  == test["match_id"].nunique()
print("[OK] Match-level canonical berhasil dibangun.")

train_match: (39386, 72)
test_match : (21211, 20)
[OK] Match-level canonical berhasil dibangun.


In [12]:
display(train_match.head(3))
display(test_match.head(3))

,match_id,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel,team_a_goals,team_b_goals,team_a_elo,team_b_elo,team_a_rank,team_b_rank,team_a_rank_missing,team_b_rank_missing,team_a_points_last5,team_b_points_last5,...,team_a_opp_elo,team_b_opp_elo,team_a_opp_rank,team_b_opp_rank,team_a_opp_rank_missing,team_b_opp_rank_missing,team_a_opp_points_last5,team_b_opp_points_last5,team_a_opp_gd_last5,team_b_opp_gd_last5,team_a_opp_points_last10,team_b_opp_points_last10,team_a_opp_avg_goals_last5,team_b_opp_avg_goals_last5,team_a_opp_avg_conceded_last5,team_b_opp_avg_conceded_last5,team_a_opp_win_rate_last10,team_b_opp_win_rate_last10,team_a_opp_days_since_last_match,team_b_opp_days_since_last_match,team_a_points_last5_diff,team_b_points_last5_diff,team_a_gd_last5_diff,team_b_gd_last5_diff,team_a_rank_diff,team_b_rank_diff,team_a_h2h_points_last5,team_b_h2h_points_last5,team_a_h2h_gd_last5,team_b_h2h_gd_last5
0,M000001,1872-11-30,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1500.0000,1500.0000,NaN,NaN,1,1,NaN,NaN,...,1500.0000,1500.0000,NaN,NaN,1,1,1.0000,NaN,0.0000,NaN,1.0000,NaN,0.0000,NaN,0.0000,NaN,0.0000,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000002,1873-03-08,M,Friendly,England,0,NaN,NaN,England,Scotland,1,0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,4,2,1500.0000,1484.0000,NaN,NaN,1,1,1.0000,1.0000,...,1500.0000,1516.0000,NaN,NaN,1,1,1.0000,4.0000,0.0000,2.0000,1.0000,4.0000,0.0000,2.0000,0.0000,1.0000,0.0000,0.5000,98.0000,0.0000,0.0000,-3.0000,0.0000,-2.0000,NaN,NaN,1.0000,1.0000,0.0000,0.0000
2,M000003,1874-03-07,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,1,2,1498.5305,1484.0000,NaN,NaN,1,1,4.0000,1.0000,...,1501.4695,1516.0000,NaN,NaN,1,1,4.0000,4.0000,-1.0000,2.0000,4.0000,4.0000,1.3333,2.0000,1.6667,1.0000,0.3333,0.5000,0.0000,364.0000,0.0000,-3.0000,3.0000,-4.0000,NaN,NaN,4.0000,1.0000,2.0000,-2.0000


,match_id,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel
0,M034984,2011-08-06,M,Indian Ocean Island Games,Seychelles,0,NaN,25.7902,Mauritius,Seychelles,0,1,CAF,CAF,1283330.0000,92409.0000,9197.0270,12189.0952,1751.8957,0.0000
1,M034985,2011-08-06,M,Indian Ocean Island Games,Seychelles,1,NaN,25.7902,Comoros,Maldives,1,0,CAF,AFC,656024.0000,361575.0000,1447.9451,7291.4660,1517.0107,5483.1175
2,M034986,2011-08-06,M,Indian Ocean Island Games,Seychelles,1,NaN,25.7902,Madagascar,Réunion,0,1,CAF,Unknown,21731053.0000,NaN,531.2654,NaN,NaN,NaN


Canonicalization tetap sama dengan EXP 00 (urut alfabet) agar hasil antar eksperimen konsisten.

## 06. Feature Engineering Match-Level

Prinsip feature engineering EXP 01:
1. **Known-future only** — semua fitur bisa diketahui saat inference
2. **Symmetry-aware** — fitur disusun dengan _a/_b dan ditambah selisih/rasio
3. **Leakage-safe** — tidak ada target encoding atau statistik agregat dari seluruh data

In [13]:
def engineer_match_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tambahkan fitur turunan ke DataFrame match-level.
    Bekerja identik untuk train_match dan test_match."""
    df = df.copy()

    # --- 1. Date features ---
    dt = pd.to_datetime(df["date"])
    df["match_year"]      = dt.dt.year
    df["match_month"]     = dt.dt.month
    df["match_quarter"]   = dt.dt.quarter
    df["match_dayofweek"] = dt.dt.dayofweek
    df["match_dayofyear"] = dt.dt.dayofyear
    df["match_is_weekend"]= (dt.dt.dayofweek >= 5).astype(int)
    df["match_decade"]    = (dt.dt.year // 10) * 10

    # --- 2. Home / venue / confederation ---
    df["home_side"] = np.where(
        df["team_a_is_home"] == 1, 1,
        np.where(df["team_b_is_home"] == 1, -1, 0)
    )
    conf_a = df["team_a_confederation"].fillna("UNKNOWN").astype(str)
    conf_b = df["team_b_confederation"].fillna("UNKNOWN").astype(str)
    df["same_confederation"] = (conf_a == conf_b).astype(int)

    t_low = df["tournament"].str.lower().fillna("")
    df["is_friendly"]       = t_low.str.contains("friendly").astype(int)
    df["is_world_cup"]      = t_low.str.contains("world cup").astype(int)
    df["is_qualification"]  = t_low.str.contains("qualif").astype(int)
    df["is_nations_league"] = t_low.str.contains("nations league").astype(int)
    df["tournament_weight_proxy"] = df["tournament"].apply(
        lambda x: get_tournament_weight(str(x))
    )

    # --- 3. Numeric symmetry features ---
    pairs = [
        ("population", "team_a_population", "team_b_population"),
        ("gdp",        "team_a_gdp_per_capita", "team_b_gdp_per_capita"),
        ("distance",   "team_a_distance_travel", "team_b_distance_travel"),
    ]
    for prefix, col_a, col_b in pairs:
        a = df[col_a].astype(float)
        b = df[col_b].astype(float)
        df[f"{prefix}_diff"]     = a - b
        df[f"{prefix}_abs_diff"] = np.abs(a - b)
        df[f"log_{prefix}_a"]    = np.log1p(np.abs(a))
        df[f"log_{prefix}_b"]    = np.log1p(np.abs(b))
        df[f"log_{prefix}_diff"] = np.log1p(np.abs(a)) - np.log1p(np.abs(b))
        # Safe ratio
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where((b == 0) | b.isna(), np.nan, a / b)
        # Guard inf
        ratio = np.where(np.isinf(ratio), np.nan, ratio)
        df[f"{prefix}_ratio_ab"] = ratio

    # --- 4. Matchup category features ---
    df["pair_key"] = df["team_a"].astype(str) + "__VS__" + df["team_b"].astype(str)
    df["confed_pair_key"] = (
        df["team_a_confederation"].fillna("UNK").astype(str)
        + "__VS__"
        + df["team_b_confederation"].fillna("UNK").astype(str)
    )

    return df


# Terapkan
train_match_fe = engineer_match_features(train_match)
test_match_fe  = engineer_match_features(test_match)

print(f"train_match_fe: {train_match_fe.shape}")
print(f"test_match_fe : {test_match_fe.shape}")
print(f"\nKolom baru: {train_match_fe.shape[1] - train_match.shape[1]} fitur ditambahkan")

train_match_fe: (39386, 106)
test_match_fe : (21211, 54)

Kolom baru: 34 fitur ditambahkan


### Catatan Feature Engineering
- Semua fitur **known-future** dan **test-feasible**
- Fitur numerik simetris (population, GDP, distance) diperluas dengan diff, abs_diff, log, dan safe ratio
- Fitur matchup (pair_key, confed_pair_key) dibuat sebagai kategorikal untuk CatBoost
- Tidak ada target encoding — menghindari leakage dan masalah unseen teams

## 07. Final Feature Set Definition

In [14]:
categorical_features = [
    "team_a", "team_b",
    "gender", "tournament", "venue_country",
    "team_a_confederation", "team_b_confederation",
    "pair_key", "confed_pair_key",
]

numeric_features = [
    # date
    "match_year", "match_month", "match_quarter",
    "match_dayofweek", "match_dayofyear", "match_is_weekend", "match_decade",
    # home / venue
    "neutral", "team_a_is_home", "team_b_is_home", "home_side",
    "same_confederation",
    # tournament flags
    "is_friendly", "is_world_cup", "is_qualification", "is_nations_league",
    "tournament_weight_proxy",
    # venue
    "altitude_venue", "temperature_venue",
    # population
    "team_a_population", "team_b_population",
    "population_diff", "population_abs_diff",
    "log_population_a", "log_population_b", "log_population_diff",
    "population_ratio_ab",
    # gdp
    "team_a_gdp_per_capita", "team_b_gdp_per_capita",
    "gdp_diff", "gdp_abs_diff",
    "log_gdp_a", "log_gdp_b", "log_gdp_diff",
    "gdp_ratio_ab",
    # distance
    "team_a_distance_travel", "team_b_distance_travel",
    "distance_diff", "distance_abs_diff",
    "log_distance_a", "log_distance_b", "log_distance_diff",
    "distance_ratio_ab",
]

ALL_FEATURES = categorical_features + numeric_features
print(f"Categorical features: {len(categorical_features)}")
print(f"Numeric features:     {len(numeric_features)}")
print(f"Total features:       {len(ALL_FEATURES)}")

Categorical features: 9
Numeric features:     43
Total features:       52


In [15]:
# Simpan daftar fitur ke JSON
feature_info = {
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "total_features": len(ALL_FEATURES),
}
with open(f"{SUM_DIR}/feature_columns.json", "w") as f:
    json.dump(feature_info, f, indent=2)
print(f"[OK] Feature list disimpan ke {SUM_DIR}/feature_columns.json")

[OK] Feature list disimpan ke ../outputs/exp01_shared_feature_catboost_metric_aware/summaries/feature_columns.json


In [16]:
def prepare_features(df, cat_feats, num_feats):
    """Siapkan DataFrame fitur untuk CatBoost.
    Categorical → fillna + string; Numeric → biarkan NaN."""
    X = df[cat_feats + num_feats].copy()
    for col in cat_feats:
        X[col] = X[col].fillna("MISSING").astype(str)
    return X

print("[OK] prepare_features() siap.")

[OK] prepare_features() siap.


Feature set ini adalah **baseline shared-feature first** — bukan feature set final kompetisi.

## 08. Temporal Validation Split

In [17]:
train_fold, valid_fold = make_time_based_holdout(train_match_fe, valid_fraction=0.2)

# Verifikasi
t_max = train_fold["date"].max()
v_min = valid_fold["date"].min()
print(f"\nTrain fold max date: {t_max}")
print(f"Valid fold min date: {v_min}")

overlap = set(train_fold["match_id"]) & set(valid_fold["match_id"])
assert len(overlap) == 0, f"Overlap match_id: {len(overlap)}"
print(f"[OK] Tidak ada match_id overlap.")

Train fold: 31,508 matches  (1872-11-30 00:00:00 -> 2005-01-30 00:00:00)
Valid fold: 7,878 matches  (2005-02-01 00:00:00 -> 2011-08-04 00:00:00)

Train fold max date: 2005-01-30 00:00:00
Valid fold min date: 2005-02-01 00:00:00
[OK] Tidak ada match_id overlap.


Split dilakukan per **match** (bukan per row), 20% match terakhir menjadi validation.

## 09. Baseline 0 — Naïve Reference (Prediksi 1-1)

In [18]:
awmae_naive = awmae_score(
    valid_fold["team_a_goals"], valid_fold["team_b_goals"],
    np.ones(len(valid_fold)), np.ones(len(valid_fold)),
    valid_fold["tournament"],
)
print(f"Baseline 0 (Naive 1-1): AW-MAE = {awmae_naive:.6f}")

# Inisialisasi tabel perbandingan
results_table = [{"baseline": "Naive 1-1", "valid_awmae": awmae_naive}]

Baseline 0 (Naive 1-1): AW-MAE = 4.662228


Baseline naïf ini **hanya referensi** — bukan target performa.

## 10. Baseline A — Direct Goals CatBoost Regressor

Latih dua model regresi terpisah:
- `model_goal_a` → prediksi `team_a_goals`
- `model_goal_b` → prediksi `team_b_goals`

Lalu clip dan round ke integer.

In [19]:
# Siapkan data
X_train = prepare_features(train_fold, categorical_features, numeric_features)
X_valid = prepare_features(valid_fold, categorical_features, numeric_features)

y_goal_a_train = train_fold["team_a_goals"].values.astype(float)
y_goal_b_train = train_fold["team_b_goals"].values.astype(float)
y_goal_a_valid = valid_fold["team_a_goals"].values.astype(float)
y_goal_b_valid = valid_fold["team_b_goals"].values.astype(float)

train_pool_a = Pool(X_train, y_goal_a_train, cat_features=categorical_features)
valid_pool_a = Pool(X_valid, y_goal_a_valid, cat_features=categorical_features)
train_pool_b = Pool(X_train, y_goal_b_train, cat_features=categorical_features)
valid_pool_b = Pool(X_valid, y_goal_b_valid, cat_features=categorical_features)

print(f"X_train: {X_train.shape}, X_valid: {X_valid.shape}")

X_train: (31508, 52), X_valid: (7878, 52)


In [20]:
goal_reg_params = dict(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
    early_stopping_rounds=200,
)

print("Training model_goal_a ...")
model_goal_a = CatBoostRegressor(**goal_reg_params)
model_goal_a.fit(train_pool_a, eval_set=valid_pool_a)
print(f"  Best iteration: {model_goal_a.best_iteration_}")

print("\nTraining model_goal_b ...")
model_goal_b = CatBoostRegressor(**goal_reg_params)
model_goal_b.fit(train_pool_b, eval_set=valid_pool_b)
print(f"  Best iteration: {model_goal_b.best_iteration_}")

Training model_goal_a ...
0:	learn: 1.7980699	test: 1.6274934	best: 1.6274934 (0)	total: 396ms	remaining: 9m 53s
200:	learn: 1.4982779	test: 1.4465766	best: 1.4458957 (195)	total: 31.5s	remaining: 3m 23s
400:	learn: 1.4398159	test: 1.4459861	best: 1.4448915 (287)	total: 58.2s	remaining: 2m 39s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.444891519
bestIteration = 287

Shrink model to first 288 iterations.
  Best iteration: 287

Training model_goal_b ...
0:	learn: 1.7802795	test: 1.8911263	best: 1.8911263 (0)	total: 108ms	remaining: 2m 42s
200:	learn: 1.4919324	test: 1.6371448	best: 1.6371448 (200)	total: 32.6s	remaining: 3m 30s
400:	learn: 1.4374922	test: 1.6227672	best: 1.6227672 (400)	total: 59.2s	remaining: 2m 42s
600:	learn: 1.3981921	test: 1.6137263	best: 1.6136456 (598)	total: 1m 26s	remaining: 2m 9s
800:	learn: 1.3706609	test: 1.6095486	best: 1.6095258 (799)	total: 1m 53s	remaining: 1m 38s
1000:	learn: 1.3447814	test: 1.6065780	best: 1.6065158 (990)	total

In [21]:
# Prediksi kontinu
MAX_GOALS = 7
pred_goal_a_valid = model_goal_a.predict(X_valid)
pred_goal_b_valid = model_goal_b.predict(X_valid)

# Clip dan round
pred_a_direct = np.clip(np.round(pred_goal_a_valid), 0, MAX_GOALS).astype(int)
pred_b_direct = np.clip(np.round(pred_goal_b_valid), 0, MAX_GOALS).astype(int)

awmae_direct = awmae_score(
    y_goal_a_valid, y_goal_b_valid,
    pred_a_direct, pred_b_direct,
    valid_fold["tournament"],
)
print(f"Baseline A (Direct + Round): AW-MAE = {awmae_direct:.6f}")
results_table.append({"baseline": "Direct Regression", "valid_awmae": awmae_direct})

Baseline A (Direct + Round): AW-MAE = 3.210298


### Catatan Baseline A
- Sederhana dan feasible
- Clip ke `[0, MAX_GOALS]` lalu round ke integer
- Belum metric-aware — rounding polos bisa salah outcome

## 11. Target Decomposition untuk Baseline B

In [22]:
def build_outcome_target(goal_a, goal_b):
    """Outcome class: 0=team_a_win, 1=draw, 2=team_b_win."""
    return np.where(goal_a > goal_b, 0, np.where(goal_a == goal_b, 1, 2))

# Bangun target dekomposisi
y_total_train   = (y_goal_a_train + y_goal_b_train).astype(float)
y_gd_train      = (y_goal_a_train - y_goal_b_train).astype(float)
y_outcome_train = build_outcome_target(y_goal_a_train, y_goal_b_train).astype(int)

y_total_valid   = (y_goal_a_valid + y_goal_b_valid).astype(float)
y_gd_valid      = (y_goal_a_valid - y_goal_b_valid).astype(float)
y_outcome_valid = build_outcome_target(y_goal_a_valid, y_goal_b_valid).astype(int)

print("Distribusi outcome class (train fold):")
for cls, name in [(0,"team_a_win"),(1,"draw"),(2,"team_b_win")]:
    n = (y_outcome_train == cls).sum()
    print(f"  {cls} ({name}): {n:,} ({n/len(y_outcome_train)*100:.1f}%)")

Distribusi outcome class (train fold):
  0 (team_a_win): 12,297 (39.0%)
  1 (draw): 6,846 (21.7%)
  2 (team_b_win): 12,365 (39.2%)


Decomposition ini lebih dekat ke struktur AW-MAE: outcome, goal difference, dan total goals adalah komponen yang langsung memengaruhi penalty metrik.

## 12. Baseline B — Outcome + Goal Difference + Total Goals Models

Latih 3 model tambahan:
1. `model_outcome` (Classifier → probabilitas outcome)
2. `model_goal_diff` (Regressor)
3. `model_total_goals` (Regressor)

In [23]:
outcome_clf_params = dict(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
    early_stopping_rounds=200,
)

print("Training model_outcome (classifier) ...")
train_pool_out = Pool(X_train, y_outcome_train, cat_features=categorical_features)
valid_pool_out = Pool(X_valid, y_outcome_valid, cat_features=categorical_features)
model_outcome = CatBoostClassifier(**outcome_clf_params)
model_outcome.fit(train_pool_out, eval_set=valid_pool_out)
print(f"  Best iteration: {model_outcome.best_iteration_}")

Training model_outcome (classifier) ...
0:	learn: 1.0912669	test: 1.0916064	best: 1.0916064 (0)	total: 188ms	remaining: 4m 41s
200:	learn: 0.8928742	test: 0.9161546	best: 0.9161546 (200)	total: 43.5s	remaining: 4m 41s
400:	learn: 0.8560815	test: 0.9074971	best: 0.9074774 (398)	total: 1m 30s	remaining: 4m 6s
600:	learn: 0.8242008	test: 0.9038880	best: 0.9038880 (600)	total: 2m 9s	remaining: 3m 14s
800:	learn: 0.7973175	test: 0.9033938	best: 0.9033705 (749)	total: 2m 50s	remaining: 2m 28s
1000:	learn: 0.7740347	test: 0.9032608	best: 0.9030090 (908)	total: 3m 32s	remaining: 1m 45s
1200:	learn: 0.7522739	test: 0.9029920	best: 0.9029394 (1085)	total: 4m 13s	remaining: 1m 3s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.9029394448
bestIteration = 1085

Shrink model to first 1086 iterations.
  Best iteration: 1085


In [24]:
# Goal diff regressor
print("Training model_goal_diff ...")
train_pool_gd = Pool(X_train, y_gd_train, cat_features=categorical_features)
valid_pool_gd = Pool(X_valid, y_gd_valid, cat_features=categorical_features)
model_goal_diff = CatBoostRegressor(**goal_reg_params)
model_goal_diff.fit(train_pool_gd, eval_set=valid_pool_gd)
print(f"  Best iteration: {model_goal_diff.best_iteration_}")

# Total goals regressor
print("\nTraining model_total_goals ...")
train_pool_tg = Pool(X_train, y_total_train, cat_features=categorical_features)
valid_pool_tg = Pool(X_valid, y_total_valid, cat_features=categorical_features)
model_total_goals = CatBoostRegressor(**goal_reg_params)
model_total_goals.fit(train_pool_tg, eval_set=valid_pool_tg)
print(f"  Best iteration: {model_total_goals.best_iteration_}")

Training model_goal_diff ...
0:	learn: 2.7496751	test: 2.7685538	best: 2.7685538 (0)	total: 125ms	remaining: 3m 7s
200:	learn: 2.2030564	test: 2.2968568	best: 2.2968433 (199)	total: 25.8s	remaining: 2m 46s
400:	learn: 2.1047185	test: 2.2777946	best: 2.2774583 (389)	total: 1m 5s	remaining: 2m 58s
600:	learn: 2.0314146	test: 2.2706165	best: 2.2696821 (586)	total: 1m 48s	remaining: 2m 42s
800:	learn: 1.9765598	test: 2.2659494	best: 2.2650382 (789)	total: 2m 34s	remaining: 2m 14s
1000:	learn: 1.9322764	test: 2.2642045	best: 2.2638167 (984)	total: 3m 7s	remaining: 1m 33s
1200:	learn: 1.8931076	test: 2.2614635	best: 2.2613659 (1192)	total: 3m 48s	remaining: 56.9s
1400:	learn: 1.8554904	test: 2.2610794	best: 2.2610659 (1399)	total: 4m 32s	remaining: 19.3s
1499:	learn: 1.8399731	test: 2.2607901	best: 2.2606422 (1468)	total: 4m 46s	remaining: 0us

bestTest = 2.260642166
bestIteration = 1468

Shrink model to first 1469 iterations.
  Best iteration: 1468

Training model_total_goals ...
0:	learn: 

In [25]:
# Prediksi validation
pred_outcome_proba_valid = model_outcome.predict_proba(X_valid)
pred_gd_valid    = model_goal_diff.predict(X_valid)
pred_total_valid = model_total_goals.predict(X_valid)

print(f"pred_outcome_proba shape: {pred_outcome_proba_valid.shape}")
print(f"pred_gd range:   [{pred_gd_valid.min():.2f}, {pred_gd_valid.max():.2f}]")
print(f"pred_total range: [{pred_total_valid.min():.2f}, {pred_total_valid.max():.2f}]")

pred_outcome_proba shape: (7878, 3)
pred_gd range:   [-12.13, 12.10]
pred_total range: [1.87, 10.04]


Yang digunakan dari classifier adalah `predict_proba`, bukan class final — karena decoder membutuhkan probabilitas outcome.

## 13. Scoreline Prior dari Train Fold

Prior realistis bahwa scoreline rendah (1-0, 1-1, 0-0) lebih umum dibanding scoreline ekstrem (5-4, 7-0).

In [26]:
def build_scoreline_prior(goal_a, goal_b, max_goals, alpha=1.0):
    """Buat smoothed probability distribution atas grid scoreline.
    alpha = Laplace smoothing constant."""
    counts = Counter(zip(
        np.clip(np.asarray(goal_a, dtype=int), 0, max_goals),
        np.clip(np.asarray(goal_b, dtype=int), 0, max_goals),
    ))
    total = sum(counts.values()) + alpha * (max_goals + 1) ** 2
    prior = {}
    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            prior[(a, b)] = (counts.get((a, b), 0) + alpha) / total
    return prior

scoreline_prior_train = build_scoreline_prior(
    train_fold["team_a_goals"], train_fold["team_b_goals"], MAX_GOALS
)

# Tampilkan top scorelines
top_sl = sorted(scoreline_prior_train.items(), key=lambda x: -x[1])[:10]
print("Top 10 scoreline prior (train fold):")
for (a, b), p in top_sl:
    print(f"  {a}-{b}: {p:.4f}")

Top 10 scoreline prior (train fold):
  1-1: 0.0944
  1-0: 0.0791
  0-1: 0.0790
  0-0: 0.0726
  1-2: 0.0629
  2-1: 0.0611
  0-2: 0.0587
  2-0: 0.0572
  2-2: 0.0396
  0-3: 0.0352


Prior ini hanya dari **train fold**, sehingga tetap leakage-safe.

## 14. Joint Integer Decoder

Mengubah output model kontinu/probabilistik menjadi pasangan integer skor final.

**Cost function** untuk kandidat (a, b):

cost = w_direct × (|a − ĝ_a| + |b − ĝ_b|)
     + w_total × |(a+b) − total̂|
     + w_gd × |(a−b) − gd̂|
     + w_outcome × (−log P(outcome(a,b)))
     + w_prior × (−log π(a,b))

Pilih kandidat dengan cost minimum.

In [27]:
def decode_single_match_score(
    pred_goal_a, pred_goal_b, pred_total, pred_gd, pred_outcome_proba,
    scoreline_prior, max_goals,
    w_direct=1.0, w_total=1.0, w_gd=1.0, w_outcome=1.0, w_prior=0.2,
    eps=1e-9,
):
    """Decode skor satu pertandingan dari output model."""
    best_cost, best_a, best_b = float("inf"), 0, 0
    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            cost = w_direct * (abs(a - pred_goal_a) + abs(b - pred_goal_b))
            cost += w_total * abs((a + b) - pred_total)
            cost += w_gd * abs((a - b) - pred_gd)
            oi = 0 if a > b else (1 if a == b else 2)
            cost += w_outcome * (-math.log(pred_outcome_proba[oi] + eps))
            pp = scoreline_prior.get((a, b), eps)
            cost += w_prior * (-math.log(pp + eps))
            if cost < best_cost:
                best_cost, best_a, best_b = cost, a, b
    return best_a, best_b


def decode_batch_scores(
    pred_ga, pred_gb, pred_total, pred_gd, pred_outcome_proba,
    scoreline_prior, max_goals,
    w_direct=1.0, w_total=1.0, w_gd=1.0, w_outcome=1.0, w_prior=0.2,
    eps=1e-9,
):
    """Decode skor batch pertandingan (vectorized)."""
    n_g = max_goals + 1
    cand_a = np.arange(n_g).repeat(n_g)
    cand_b = np.tile(np.arange(n_g), n_g)
    cand_oc = np.where(cand_a > cand_b, 0, np.where(cand_a == cand_b, 1, 2))
    cand_pr = np.array([scoreline_prior.get((int(a), int(b)), eps)
                        for a, b in zip(cand_a, cand_b)])

    ga = np.asarray(pred_ga, dtype=float).reshape(-1, 1)
    gb = np.asarray(pred_gb, dtype=float).reshape(-1, 1)
    gt = np.asarray(pred_total, dtype=float).reshape(-1, 1)
    gd = np.asarray(pred_gd, dtype=float).reshape(-1, 1)
    op = np.asarray(pred_outcome_proba, dtype=float)

    C  = w_direct * (np.abs(cand_a - ga) + np.abs(cand_b - gb))
    C += w_total  * np.abs((cand_a + cand_b) - gt)
    C += w_gd     * np.abs((cand_a - cand_b) - gd)
    C += w_outcome * (-np.log(op[:, cand_oc] + eps))
    C += w_prior  * (-np.log(cand_pr + eps))[None, :]

    best_idx = np.argmin(C, axis=1)
    return cand_a[best_idx], cand_b[best_idx]

print("[OK] Decoder functions siap.")

[OK] Decoder functions siap.


## 15. Tuning Ringan Decoder di Validation

Grid search kecil pada weight decoder. Yang dituning di EXP 01 baru **decision layer**, belum seluruh modeling pipeline.

In [28]:
# Grid search
MAX_GOALS_OPTIONS = [5, 6, 7]
W_DIRECT  = [0.5, 1.0]
W_TOTAL   = [1.0, 1.5, 2.0]
W_GD      = [1.0, 1.5, 2.0]
W_OUTCOME = [0.5, 1.0, 2.0]
W_PRIOR   = [0.0, 0.2, 0.5]

eps = 1e-9

# Pre-compute tournament weights untuk vectorized evaluation
valid_tournament_weights = np.array([
    get_tournament_weight(str(t)) for t in valid_fold["tournament"].values
])
y_true_a = y_goal_a_valid
y_true_b = y_goal_b_valid

grid_results = []
total_combos = len(MAX_GOALS_OPTIONS) * len(W_DIRECT) * len(W_TOTAL) * len(W_GD) * len(W_OUTCOME) * len(W_PRIOR)
print(f"Total kombinasi: {total_combos}")

combo_idx = 0
for max_g in MAX_GOALS_OPTIONS:
    # Build candidate grid
    n_g = max_g + 1
    cand_a = np.arange(n_g).repeat(n_g)
    cand_b = np.tile(np.arange(n_g), n_g)
    cand_oc = np.where(cand_a > cand_b, 0, np.where(cand_a == cand_b, 1, 2))

    prior = build_scoreline_prior(
        train_fold["team_a_goals"], train_fold["team_b_goals"], max_g
    )
    cand_pr = np.array([prior.get((int(a), int(b)), eps)
                        for a, b in zip(cand_a, cand_b)])

    # Pre-compute cost matrices (n_valid x n_candidates)
    ga = pred_goal_a_valid.reshape(-1, 1)
    gb = pred_goal_b_valid.reshape(-1, 1)
    gt = pred_total_valid.reshape(-1, 1)
    gd = pred_gd_valid.reshape(-1, 1)

    C_direct  = np.abs(cand_a - ga) + np.abs(cand_b - gb)
    C_total   = np.abs((cand_a + cand_b) - gt)
    C_gd_mat  = np.abs((cand_a - cand_b) - gd)
    C_outcome = -np.log(pred_outcome_proba_valid[:, cand_oc] + eps)
    C_prior   = -np.log(cand_pr + eps)[None, :]

    for w_d in W_DIRECT:
        for w_t in W_TOTAL:
            for w_g in W_GD:
                for w_o in W_OUTCOME:
                    for w_p in W_PRIOR:
                        total_cost = (w_d * C_direct + w_t * C_total +
                                      w_g * C_gd_mat + w_o * C_outcome +
                                      w_p * C_prior)
                        best_idx = np.argmin(total_cost, axis=1)
                        dec_a = cand_a[best_idx].astype(float)
                        dec_b = cand_b[best_idx].astype(float)
                        score = awmae_score_fast(
                            y_true_a, y_true_b, dec_a, dec_b,
                            valid_tournament_weights,
                        )
                        grid_results.append({
                            "max_goals": max_g,
                            "w_direct": w_d, "w_total": w_t,
                            "w_gd": w_g, "w_outcome": w_o, "w_prior": w_p,
                            "awmae": score,
                        })
                        combo_idx += 1

    print(f"  MAX_GOALS={max_g} done ({combo_idx}/{total_combos})")

grid_df = pd.DataFrame(grid_results).sort_values("awmae")
print(f"\nGrid search selesai. Total kombinasi: {len(grid_df)}")

Total kombinasi: 486
  MAX_GOALS=5 done (162/486)
  MAX_GOALS=6 done (324/486)
  MAX_GOALS=7 done (486/486)

Grid search selesai. Total kombinasi: 486


In [29]:
# Top 10 kombinasi terbaik
print("=== Top 10 Kombinasi Decoder ===")
display(grid_df.head(10))

# Best combo
best_row = grid_df.iloc[0]
best_decoder_params = {
    "max_goals": int(best_row["max_goals"]),
    "w_direct":  best_row["w_direct"],
    "w_total":   best_row["w_total"],
    "w_gd":      best_row["w_gd"],
    "w_outcome": best_row["w_outcome"],
    "w_prior":   best_row["w_prior"],
}
awmae_decoder = best_row["awmae"]
print(f"\nBest decoder AW-MAE: {awmae_decoder:.6f}")
print(f"Best params: {best_decoder_params}")

results_table.append({"baseline": "Decomp + Decoder", "valid_awmae": awmae_decoder})

=== Top 10 Kombinasi Decoder ===


,max_goals,w_direct,w_total,w_gd,w_outcome,w_prior,awmae
251,6,1.0000,1.0000,1.0000,2.0000,0.5000,3.1080
413,7,1.0000,1.0000,1.0000,2.0000,0.5000,3.1094
89,5,1.0000,1.0000,1.0000,2.0000,0.5000,3.1100
170,6,0.5000,1.0000,1.0000,2.0000,0.5000,3.1115
8,5,0.5000,1.0000,1.0000,2.0000,0.5000,3.1131
332,7,0.5000,1.0000,1.0000,2.0000,0.5000,3.1133
188,6,0.5000,1.0000,2.0000,2.0000,0.5000,3.1146
260,6,1.0000,1.0000,1.5000,2.0000,0.5000,3.1147
98,5,1.0000,1.0000,1.5000,2.0000,0.5000,3.1152
26,5,0.5000,1.0000,2.0000,2.0000,0.5000,3.1152



Best decoder AW-MAE: 3.107973
Best params: {'max_goals': 6, 'w_direct': np.float64(1.0), 'w_total': np.float64(1.0), 'w_gd': np.float64(1.0), 'w_outcome': np.float64(2.0), 'w_prior': np.float64(0.5)}


In [30]:
# Simpan grid results
grid_df.to_csv(f"{SUM_DIR}/decoder_grid_results.csv", index=False)
print(f"[OK] Grid results disimpan ke {SUM_DIR}/decoder_grid_results.csv")

[OK] Grid results disimpan ke ../outputs/exp01_shared_feature_catboost_metric_aware/summaries/decoder_grid_results.csv


## 16. Perbandingan Hasil Baseline

In [31]:
# Tabel ringkasan
results_df = pd.DataFrame(results_table).sort_values("valid_awmae")
print("=" * 55)
print("RINGKASAN PERBANDINGAN BASELINE")
print("=" * 55)
display(results_df)

# Tentukan winner
best_approach = results_df.iloc[0]["baseline"]
best_awmae = results_df.iloc[0]["valid_awmae"]
print(f"\n>>> BASELINE TERBAIK: {best_approach} (AW-MAE = {best_awmae:.6f})")

RINGKASAN PERBANDINGAN BASELINE


,baseline,valid_awmae
2,Decomp + Decoder,3.1080
1,Direct Regression,3.2103
0,Naive 1-1,4.6622



>>> BASELINE TERBAIK: Decomp + Decoder (AW-MAE = 3.107973)


In [32]:
# Decode validation dengan best params untuk analisis lanjut
if best_approach == "Decomp + Decoder":
    bp = best_decoder_params
    final_pred_a_valid, final_pred_b_valid = decode_batch_scores(
        pred_goal_a_valid, pred_goal_b_valid,
        pred_total_valid, pred_gd_valid,
        pred_outcome_proba_valid,
        build_scoreline_prior(
            train_fold["team_a_goals"], train_fold["team_b_goals"],
            bp["max_goals"]
        ),
        **bp,
    )
else:
    final_pred_a_valid = pred_a_direct.astype(float)
    final_pred_b_valid = pred_b_direct.astype(float)

# Simpan prediksi validation
valid_pred_df = valid_fold[["match_id","team_a","team_b","tournament",
                            "team_a_goals","team_b_goals"]].copy()
valid_pred_df["pred_team_a_goals"] = final_pred_a_valid.astype(int)
valid_pred_df["pred_team_b_goals"] = final_pred_b_valid.astype(int)
valid_pred_df.to_csv(f"{PRED_DIR}/valid_pred_match.csv", index=False)
print(f"[OK] Prediksi validation disimpan.")

[OK] Prediksi validation disimpan.


In [33]:
# Contoh prediksi vs aktual
print("=== Contoh Prediksi vs Aktual ===")
sample = valid_pred_df.head(15).copy()
sample["loss"] = [
    official_match_loss(r["team_a_goals"], r["team_b_goals"],
                        r["pred_team_a_goals"], r["pred_team_b_goals"])
    for _, r in sample.iterrows()
]
display(sample)

=== Contoh Prediksi vs Aktual ===


,match_id,team_a,team_b,tournament,team_a_goals,team_b_goals,pred_team_a_goals,pred_team_b_goals,loss
0,W002556,Australia,Russia,Four Nations Tournament,5,0,1,2,13.0749
1,W002557,China PR,Germany,Four Nations Tournament,0,2,1,1,4.0720
2,M028954,Haiti,Trinidad and Tobago,Friendly,0,1,1,2,1.4822
3,M028959,Bahrain,Lebanon,Friendly,2,1,2,1,0.0000
4,M028958,Hungary,Saudi Arabia,Friendly,0,0,2,1,5.9947
5,M028957,Bosnia and Herzegovina,Iran,Friendly,1,2,1,2,0.0000
6,M028955,Kuwait,North Korea,Friendly,0,0,2,1,5.9947
7,M028956,Japan,Syria,Friendly,3,0,2,1,1.7460
8,M028960,Haiti,Trinidad and Tobago,Friendly,1,2,1,2,0.0000
9,M028961,Egypt,South Korea,Friendly,1,0,1,1,2.4150


In [34]:
# Bar chart perbandingan
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#DD8452", "#4C72B0", "#55A868"]
bars = ax.bar(results_df["baseline"], results_df["valid_awmae"], color=colors[:len(results_df)])
ax.set_ylabel("AW-MAE (Validation)")
ax.set_title("Perbandingan Baseline — EXP 01")
for bar, val in zip(bars, results_df["valid_awmae"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/valid_awmae_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Figure disimpan.")

[OK] Figure disimpan.


In [35]:
# Distribusi scoreline prediksi vs aktual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

actual_sl = [f"{int(a)}-{int(b)}" for a, b in
             zip(valid_fold["team_a_goals"], valid_fold["team_b_goals"])]
pred_sl   = [f"{int(a)}-{int(b)}" for a, b in
             zip(final_pred_a_valid, final_pred_b_valid)]

top_actual = Counter(actual_sl).most_common(12)
top_pred   = Counter(pred_sl).most_common(12)

axes[0].barh([s for s,_ in top_actual], [c for _,c in top_actual], color="#4C72B0")
axes[0].set_title("Actual Scorelines (Validation)")
axes[0].invert_yaxis()

axes[1].barh([s for s,_ in top_pred], [c for _,c in top_pred], color="#55A868")
axes[1].set_title("Predicted Scorelines (Best Baseline)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/scoreline_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretasi
- Baseline terbaik dipilih berdasarkan **AW-MAE validation terendah**.
- Decoder membantu karena metrik kompetisi sangat menghukum kesalahan outcome dan goal difference.

## 17. Feature Importance dan Error Analysis Ringan

In [36]:
# Feature importance dari model outcome
fi_outcome = model_outcome.get_feature_importance()
fi_df_outcome = pd.DataFrame({
    "feature": ALL_FEATURES,
    "importance": fi_outcome,
}).sort_values("importance", ascending=False)

print("=== Top 15 Features (Outcome Classifier) ===")
display(fi_df_outcome.head(15))

fig, ax = plt.subplots(figsize=(10, 6))
top_fi = fi_df_outcome.head(15)
ax.barh(top_fi["feature"], top_fi["importance"], color="#4C72B0")
ax.set_xlabel("Importance")
ax.set_title("Top 15 Feature Importance — Outcome Classifier")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/top_feature_importance_outcome.png", dpi=150, bbox_inches="tight")
plt.show()

=== Top 15 Features (Outcome Classifier) ===


,feature,importance
1,team_b,10.8955
0,team_a,10.3855
7,pair_key,7.1378
4,venue_country,6.8867
3,tournament,5.8520
9,match_year,3.5094
6,team_b_confederation,3.3748
8,confed_pair_key,3.3178
5,team_a_confederation,3.2368
17,team_a_is_home,2.9735


In [37]:
# Feature importance dari total goals regressor
fi_total = model_total_goals.get_feature_importance()
fi_df_total = pd.DataFrame({
    "feature": ALL_FEATURES,
    "importance": fi_total,
}).sort_values("importance", ascending=False)

print("=== Top 15 Features (Total Goals Regressor) ===")
display(fi_df_total.head(15))

fig, ax = plt.subplots(figsize=(10, 6))
top_fi2 = fi_df_total.head(15)
ax.barh(top_fi2["feature"], top_fi2["importance"], color="#55A868")
ax.set_xlabel("Importance")
ax.set_title("Top 15 Feature Importance — Total Goals Regressor")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/top_feature_importance_total_goals.png", dpi=150, bbox_inches="tight")
plt.show()

=== Top 15 Features (Total Goals Regressor) ===


,feature,importance
9,match_year,10.0543
2,gender,8.2729
1,team_b,7.1627
0,team_a,7.1612
3,tournament,6.5885
7,pair_key,5.8735
4,venue_country,4.6511
6,team_b_confederation,4.6301
15,match_decade,4.2318
5,team_a_confederation,3.8030


In [38]:
# Error analysis per kategori
def per_category_awmae(valid_df, pred_a, pred_b, col, top_n=10):
    """Hitung AW-MAE per kategori."""
    valid_df = valid_df.copy()
    valid_df["_pa"] = pred_a
    valid_df["_pb"] = pred_b
    groups = valid_df.groupby(col)
    records = []
    for name, grp in groups:
        if len(grp) < 5:
            continue
        score = awmae_score(
            grp["team_a_goals"], grp["team_b_goals"],
            grp["_pa"], grp["_pb"], grp["tournament"],
        )
        records.append({"category": name, "awmae": score, "n_matches": len(grp)})
    return pd.DataFrame(records).sort_values("awmae", ascending=False).head(top_n)

print("=== AW-MAE per Gender ===")
display(per_category_awmae(valid_fold, final_pred_a_valid, final_pred_b_valid, "gender"))

print("\n=== AW-MAE per Neutral ===")
display(per_category_awmae(valid_fold, final_pred_a_valid, final_pred_b_valid, "neutral"))

=== AW-MAE per Gender ===


,category,awmae,n_matches
1,W,3.8828,1848
0,M,2.8539,6030



=== AW-MAE per Neutral ===


,category,awmae,n_matches
1,1,3.5597,2583
0,0,2.9068,5295


In [39]:
# AW-MAE per top tournament
print("=== AW-MAE per Top Tournament ===")
display(per_category_awmae(valid_fold, final_pred_a_valid, final_pred_b_valid,
                           "tournament", top_n=15))

=== AW-MAE per Top Tournament ===


,category,awmae,n_matches
62,SAFF Championship,7.9020,15
71,Viva World Cup,7.1805,42
59,Pacific Games,6.4455,20
56,OFC Championship,6.2044,22
32,Coupe de l'Outre-Mer,6.1085,26
8,African Championship,5.4054,48
39,FIFI Wild Cup,5.2647,10
46,Island Games,5.1933,164
72,WAFF Championship,5.1719,55
64,South Asian Games,5.1499,10


In [40]:
# AW-MAE per predicted outcome
pred_outcome_label = np.where(
    final_pred_a_valid > final_pred_b_valid, "A_win",
    np.where(final_pred_a_valid == final_pred_b_valid, "Draw", "B_win")
)
valid_fold_tmp = valid_fold.copy()
valid_fold_tmp["pred_outcome"] = pred_outcome_label
print("=== AW-MAE per Predicted Outcome ===")
display(per_category_awmae(valid_fold_tmp, final_pred_a_valid, final_pred_b_valid,
                           "pred_outcome"))

=== AW-MAE per Predicted Outcome ===


,category,awmae,n_matches
0,A_win,3.2356,3267
2,Draw,3.0456,1367
1,B_win,3.0032,3244


In [41]:
# Top 10 pertandingan dengan loss terbesar
valid_pred_df["match_loss"] = [
    official_match_loss(r["team_a_goals"], r["team_b_goals"],
                        r["pred_team_a_goals"], r["pred_team_b_goals"])
    for _, r in valid_pred_df.iterrows()
]
print("=== Top 10 Pertandingan dengan Loss Terbesar ===")
display(valid_pred_df.nlargest(10, "match_loss")[
    ["team_a","team_b","tournament","team_a_goals","team_b_goals",
     "pred_team_a_goals","pred_team_b_goals","match_loss"]
])

=== Top 10 Pertandingan dengan Loss Terbesar ===


,team_a,team_b,tournament,team_a_goals,team_b_goals,pred_team_a_goals,pred_team_b_goals,match_loss
1881,Cameroon,South Africa,African Championship,2,24,3,2,78.2847
2025,Monaco,Sápmi,Viva World Cup,1,21,2,1,68.8595
6000,Kuwait,Palestine,WAFF Championship,0,17,4,1,64.3002
2015,Monaco,Sápmi,Viva World Cup,0,14,2,1,43.1378
7133,Nepal,Pakistan,SAFF Championship,12,0,2,3,35.4924
2384,Australia,Taiwan,AFC Olympic Qualifying Tournament,10,0,1,2,28.3612
2942,Lebanon,Syria,WAFF Championship,7,0,0,4,28.3612
99,Guam,North Korea,EAFF Championship,0,21,0,4,26.7753
6200,Guyana,Saint Lucia,CONCACAF Gold Cup qualification,8,0,2,4,25.0005
7658,Gibraltar,Isle of Man,Island Games,0,9,4,3,25.0005


### Insight Error Analysis
- Matches dengan outcome salah (outcome flip) mendapat loss sangat tinggi.
- Draw cenderung paling sulit diprediksi.
- Pertandingan berbobot tinggi (World Cup) memberikan kontribusi loss lebih besar saat error.

## 18. Retrain Model Terbaik pada Full Train Match

Setelah baseline terbaik ditentukan, retrain menggunakan **seluruh** `train_match`.
Decoder weights menggunakan best setting dari validation, diaplikasikan apa adanya ke test.

In [42]:
# Siapkan full train
X_full = prepare_features(train_match_fe, categorical_features, numeric_features)
y_ga_full = train_match_fe["team_a_goals"].values.astype(float)
y_gb_full = train_match_fe["team_b_goals"].values.astype(float)
y_total_full = (y_ga_full + y_gb_full).astype(float)
y_gd_full = (y_ga_full - y_gb_full).astype(float)
y_outcome_full = build_outcome_target(y_ga_full, y_gb_full).astype(int)

# Params tanpa early stopping (karena tidak ada eval_set)
retrain_reg_params = {k: v for k, v in goal_reg_params.items()
                      if k != "early_stopping_rounds"}
retrain_clf_params = {k: v for k, v in outcome_clf_params.items()
                      if k != "early_stopping_rounds"}

full_pool = Pool(X_full, cat_features=categorical_features)

print(f"Full train: {X_full.shape}")

Full train: (39386, 52)


In [43]:
# Retrain semua model pada full data
print("Retrain model_goal_a (full) ...")
model_goal_a_full = CatBoostRegressor(**retrain_reg_params)
model_goal_a_full.fit(Pool(X_full, y_ga_full, cat_features=categorical_features))

print("Retrain model_goal_b (full) ...")
model_goal_b_full = CatBoostRegressor(**retrain_reg_params)
model_goal_b_full.fit(Pool(X_full, y_gb_full, cat_features=categorical_features))

if best_approach == "Decomp + Decoder":
    print("Retrain model_outcome (full) ...")
    model_outcome_full = CatBoostClassifier(**retrain_clf_params)
    model_outcome_full.fit(Pool(X_full, y_outcome_full, cat_features=categorical_features))

    print("Retrain model_goal_diff (full) ...")
    model_gd_full = CatBoostRegressor(**retrain_reg_params)
    model_gd_full.fit(Pool(X_full, y_gd_full, cat_features=categorical_features))

    print("Retrain model_total_goals (full) ...")
    model_tg_full = CatBoostRegressor(**retrain_reg_params)
    model_tg_full.fit(Pool(X_full, y_total_full, cat_features=categorical_features))

    # Rebuild scoreline prior pada full train
    bp = best_decoder_params
    scoreline_prior_full = build_scoreline_prior(
        train_match_fe["team_a_goals"], train_match_fe["team_b_goals"],
        bp["max_goals"],
    )

print("[OK] Retrain selesai.")

Retrain model_goal_a (full) ...
0:	learn: 1.7653373	total: 109ms	remaining: 2m 43s
200:	learn: 1.4862180	total: 25.6s	remaining: 2m 45s
400:	learn: 1.4337859	total: 51.3s	remaining: 2m 20s
600:	learn: 1.3995723	total: 1m 36s	remaining: 2m 24s
800:	learn: 1.3708799	total: 2m 15s	remaining: 1m 57s
1000:	learn: 1.3444084	total: 2m 56s	remaining: 1m 27s
1200:	learn: 1.3197230	total: 3m 36s	remaining: 53.8s
1400:	learn: 1.2986173	total: 4m 16s	remaining: 18.1s
1499:	learn: 1.2886967	total: 4m 37s	remaining: 0us
Retrain model_goal_b (full) ...
0:	learn: 1.8026232	total: 198ms	remaining: 4m 56s
200:	learn: 1.5030075	total: 39.9s	remaining: 4m 17s
400:	learn: 1.4491378	total: 1m 11s	remaining: 3m 16s
600:	learn: 1.4110872	total: 1m 50s	remaining: 2m 45s
800:	learn: 1.3825633	total: 2m 18s	remaining: 2m
1000:	learn: 1.3604537	total: 2m 53s	remaining: 1m 26s
1200:	learn: 1.3396661	total: 3m 26s	remaining: 51.4s
1400:	learn: 1.3194646	total: 3m 57s	remaining: 16.8s
1499:	learn: 1.3100550	total: 4

## 19. Inference pada Test Match

In [44]:
# Siapkan fitur test
X_test = prepare_features(test_match_fe, categorical_features, numeric_features)
print(f"X_test: {X_test.shape}")

# Prediksi direct goals
pred_ga_test = model_goal_a_full.predict(X_test)
pred_gb_test = model_goal_b_full.predict(X_test)

X_test: (21211, 52)


In [45]:
if best_approach == "Decomp + Decoder":
    # Prediksi decomposition
    pred_out_test  = model_outcome_full.predict_proba(X_test)
    pred_gd_test   = model_gd_full.predict(X_test)
    pred_total_test = model_tg_full.predict(X_test)

    bp = best_decoder_params
    test_pred_a, test_pred_b = decode_batch_scores(
        pred_ga_test, pred_gb_test,
        pred_total_test, pred_gd_test,
        pred_out_test,
        scoreline_prior_full,
        **bp,
    )
else:
    test_pred_a = np.clip(np.round(pred_ga_test), 0, MAX_GOALS).astype(int)
    test_pred_b = np.clip(np.round(pred_gb_test), 0, MAX_GOALS).astype(int)

# Buat test prediction DataFrame
test_pred_match_df = test_match_fe[["match_id","team_a","team_b"]].copy()
test_pred_match_df["pred_team_a_goals"] = test_pred_a.astype(int)
test_pred_match_df["pred_team_b_goals"] = test_pred_b.astype(int)

print(f"test_pred_match shape: {test_pred_match_df.shape}")
display(test_pred_match_df.head())

# Simpan
test_pred_match_df.to_csv(f"{PRED_DIR}/test_pred_match.csv", index=False)
print(f"[OK] Prediksi test disimpan.")

test_pred_match shape: (21211, 5)


,match_id,team_a,team_b,pred_team_a_goals,pred_team_b_goals
0,M034984,Mauritius,Seychelles,1,1
1,M034985,Comoros,Maldives,2,1
2,M034986,Madagascar,Réunion,1,1
3,M034987,El Salvador,Venezuela,1,2
4,M034988,Mayotte,Réunion,1,2


[OK] Prediksi test disimpan.


## 20. Reverse Mapping ke Row-Level Submission

In [46]:
submission = match_predictions_to_submission(test, test_pred_match_df)
print(f"Submission shape: {submission.shape}")
display(submission.head())

Submission shape: (42422, 3)


,Id,team_goals,opp_goals
0,M034984_Seychelles,1,1
1,M034984_Mauritius,1,1
2,M034985_Comoros,2,1
3,M034985_Maldives,1,2
4,M034986_Réunion,1,1


In [47]:
# Verifikasi format
assert len(submission) == len(sample_sub), \
    f"Row mismatch: {len(submission)} vs {len(sample_sub)}"
assert (submission["Id"].values == sample_sub["Id"].values).all(), \
    "Id order mismatch!"
assert list(submission.columns) == ["Id","team_goals","opp_goals"], \
    f"Column mismatch: {list(submission.columns)}"
assert submission["team_goals"].notna().all(), "Ada NaN di team_goals!"
assert submission["opp_goals"].notna().all(),  "Ada NaN di opp_goals!"

# Pastikan integer
submission["team_goals"] = submission["team_goals"].astype(int)
submission["opp_goals"]  = submission["opp_goals"].astype(int)

print("[OK] Format submission terverifikasi.")

# Simpan
sub_path = f"{SUB_DIR}/submission_exp01_best.csv"
submission.to_csv(sub_path, index=False)
print(f"[OK] Submission disimpan ke {sub_path}")

[OK] Format submission terverifikasi.
[OK] Submission disimpan ke ../outputs/exp01_shared_feature_catboost_metric_aware/submissions/submission_exp01_best.csv


In [48]:
# Simpan summary metrics
metrics_summary = {
    "experiment": "EXP 01",
    "best_baseline": best_approach,
    "valid_awmae_naive": float(awmae_naive),
    "valid_awmae_direct": float(awmae_direct),
    "valid_awmae_decoder": float(awmae_decoder),
    "best_valid_awmae": float(best_awmae),
    "best_decoder_params": best_decoder_params if best_approach == "Decomp + Decoder" else None,
    "n_features": len(ALL_FEATURES),
    "n_cat_features": len(categorical_features),
    "n_num_features": len(numeric_features),
    "train_matches": len(train_match_fe),
    "test_matches": len(test_match_fe),
    "valid_matches": len(valid_fold),
}
with open(f"{SUM_DIR}/exp01_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

# Notes
with open(f"{SUM_DIR}/experiment_notes.txt", "w") as f:
    f.write(f"EXP 01 — Shared-Feature CatBoost Baseline\n")
    f.write(f"Best approach: {best_approach}\n")
    f.write(f"Best validation AW-MAE: {best_awmae:.6f}\n")
    f.write(f"Naive 1-1 AW-MAE: {awmae_naive:.6f}\n")
    f.write(f"Improvement over naive: {(awmae_naive - best_awmae):.6f}\n")

print(f"[OK] Metrics summary disimpan.")
print(f"\nHasil akhir:")
print(f"  Naive 1-1:          {awmae_naive:.6f}")
print(f"  Direct Regression:  {awmae_direct:.6f}")
print(f"  Decomp + Decoder:   {awmae_decoder:.6f}")
print(f"  >>> BEST: {best_approach} = {best_awmae:.6f}")

[OK] Metrics summary disimpan.

Hasil akhir:
  Naive 1-1:          4.662228
  Direct Regression:  3.210298
  Decomp + Decoder:   3.107973
  >>> BEST: Decomp + Decoder = 3.107973


## 21. Ringkasan Hasil Eksperimen EXP 01

### Temuan Utama

1. **Baseline shared-feature berhasil dibangun** — menggunakan hanya 18 kolom yang tersedia di train dan test.
2. **CatBoost** menangani categorical features (team names, tournament) dan missing values secara native.
3. **Dua pendekatan dibandingkan**: direct regression vs decomposition + metric-aware decoder.
4. **Joint integer decoder** memberi hasil lebih baik karena mempertimbangkan outcome, goal difference, dan scoreline prior secara bersamaan.
5. **Temporal validation leakage-safe** digunakan secara konsisten.
6. **Submission berhasil dibuat** dengan format yang valid.

### Keterbatasan

- Belum memanfaatkan informasi strength/rating tim.
- Feature engineering masih sederhana (belum ada temporal aggregation aman).
- Decoder tuning masih grid search kasar.
- Belum ada ensemble atau model diversity.

## 22. Next Step ke EXP 02

Eksperimen berikutnya akan mulai mengeksplorasi:

1. **Pemanfaatan train-only signal secara aman** — misalnya membangun rating/strength proxy yang bisa direkon dari data yang tersedia.
2. **Feature engineering temporal** — historical aggregation yang leakage-safe.
3. **Strategi untuk unseen teams** — default rating, confederation-based imputation.
4. **Model diversity** — LightGBM / XGBoost sebagai pembanding atau ensemble.
5. **Decoder yang lebih sophisticated** — mungkin optimasi langsung terhadap AW-MAE.

Semua pendekatan di atas harus tetap **feasible terhadap test set** — tidak bergantung pada fitur yang tidak tersedia saat inference.

---

*EXP 01 selesai. Baseline pertama yang feasible dan metric-aware telah dibangun.*